In [ ]:
# # Code to convert this notebook to .py (if you want to run it via command line or with Slurm)
# from subprocess import call
# command = "jupyter nbconvert ingest-graph.ipynb --to python"
# call(command,shell=True)

## Import Libraries/Packages/Functions

In [ ]:
import os
import subprocess
import time
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import torch
import re
import argparse
from tqdm.auto import tqdm


from huggingface_hub import hf_hub_download

from langchain_community.document_loaders import DataFrameLoader
from langchain_neo4j import Neo4jGraph
from langchain_experimental.llms.ollama_functions import OllamaFunctions
from langchain_experimental.graph_transformers import LLMGraphTransformer


import modal 
from modal import build, enter, method

# local
import utils


In [ ]:
load_dotenv(find_dotenv())
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Configurations

In [ ]:
if utils.is_interactive():
    # sample usage
    jupyter_args = "--data_repo_id=potsu-potsu/mini-bioasq-with-metadata \
                    --llm_model=mistral:instruct"
    
    jupyter_args = jupyter_args.split()
    print(jupyter_args)

    # this allows you to change functions in files like utils.py and have this notebook automatically update with your revisions
    %load_ext autoreload 
    %autoreload 2

In [ ]:
parser = argparse.ArgumentParser(description="Data Retrieval")
parser.add_argument(
    "--data_repo_id", type=str,
    help="HuggingFace repository ID where dataset is stored",
)
parser.add_argument(
    "--llm_model", type=str,
    help="model to use for extracting graph nodes and relationships",
)
parser.add_argument(
    "--use_test_set_only", action="store_true", 
    help="if True, only ingest relevant passages from the test set",
)


if utils.is_interactive():
    args = parser.parse_args(jupyter_args)
else:
    args = parser.parse_args()

# create global variables without the args prefix
for attribute_name in vars(args).keys():
    globals()[attribute_name] = getattr(args, attribute_name) 

for arg_name, arg_value in vars(args).items():
    print(f"{arg_name}: {arg_value}")


In [ ]:
corpus_file = "text-corpus/test-00000-of-00001.parquet"
train_file = "question-answer-passages/train-00000-of-00001.parquet"
test_file = "question-answer-passages/test-00000-of-00001.parquet"

## Setup Modal App

In [ ]:
# MODEL = "llama3.3"
MODEL = llm_model
GPU = "H100"
TIMEOUT = 3600

MODEL

In [ ]:
def pull_ollama_model(model: str = MODEL):
    print(f"Pulling {model}...")
    subprocess.run(["systemctl", "daemon-reload"])
    subprocess.run(["systemctl", "enable", "ollama"])
    subprocess.run(["systemctl", "start", "ollama"])
    time.sleep(2)  # 2s, wait for the service to start
    subprocess.run(["ollama", "pull", model], stdout=subprocess.PIPE)
    print(f"Finished pulling {model}.")

In [ ]:
IMAGE = (
    modal.Image.debian_slim()
    .apt_install("curl", "systemctl")
    .run_commands(  # from https://github.com/ollama/ollama/blob/main/docs/linux.md
        "curl -L https://ollama.com/download/ollama-linux-amd64.tgz -o ollama-linux-amd64.tgz",
        "tar -C /usr -xzf ollama-linux-amd64.tgz",
        "useradd -r -s /bin/false -U -m -d /usr/share/ollama ollama",
        "usermod -a -G ollama $(whoami)",
    )
    .add_local_file("ollama.service", "/etc/systemd/system/ollama.service", copy=True)
    .pip_install("pandas", "huggingface-hub", "pyarrow", "ollama", "langchain", "langchain-experimental", "langchain-community", "tqdm")
    .run_function(pull_ollama_model)
)

In [ ]:
app = modal.App(name="graph-retrieval", image=IMAGE)

In [ ]:
with IMAGE.imports():
    import pandas as pd
    import re
    import subprocess
    from huggingface_hub import hf_hub_download
    from langchain_community.document_loaders import DataFrameLoader
    from langchain_experimental.llms.ollama_functions import OllamaFunctions
    from langchain_experimental.graph_transformers import LLMGraphTransformer
    from tqdm.auto import tqdm

In [ ]:
@app.function()
def get_docs(repo_id: str, filename: str, ids: list[int] = None):
    def list_to_str(list):
        string = ', '.join(list)
        return string

    def clean_text(text):
        text = re.sub(r'\n+', ' ', text)
        text = re.sub(' +',' ',text)
        return text
    
    df_corpus = pd.read_parquet(hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset"))
    
    df_corpus["authors"] = df_corpus["authors"].apply(list_to_str)
    df_corpus["keywords"] = df_corpus["keywords"].apply(list_to_str)
    df_corpus["publish_type"] = df_corpus["publish_type"].apply(list_to_str)
    df_corpus["passage"] = df_corpus["passage"].apply(clean_text)

    if ids:
        df_corpus = df_corpus[df_corpus["id"].isin(ids)]

    corpus_loader = DataFrameLoader(df_corpus, page_content_column="passage")
    corpus_docs = corpus_loader.load()

    return corpus_docs
    
    

In [ ]:
@app.function( 
    gpu=GPU, 
    timeout=TIMEOUT,
    scaledown_window=TIMEOUT,
)
def convert_to_graph_documents(model_name: str, 
                               repo_id: str, 
                               filename: str, 
                               device: str, 
                               ids: list[int] = None, 
                               start_ndx=0, 
                               batch_size: int=200):
    
    corpus_docs = get_docs.remote(repo_id, filename, ids)

    print(f"Corpus Documents of length {len(corpus_docs)} loaded.")

    subprocess.run(["systemctl", "start", "ollama"])

    llm_graph = OllamaFunctions(
        model=model_name, 
        temperature=0, 
        format="json", 
        device=device,
    )

    llm_transformer = LLMGraphTransformer(
        llm=llm_graph, 
        node_properties=True,
        relationship_properties=True
    )

    # graph_documents = await llm_transformer.convert_to_graph_documents(documents=corpus_docs)

    graph_documents = []
    skipped_doc_ids = []  
    last_ndx = len(corpus_docs[start_ndx:]) - 1
   
    for ndx, doc in enumerate(tqdm(corpus_docs[start_ndx:])):
        try:
            graph_documents.append(llm_transformer.process_response(document=doc))
            if len(graph_documents) == batch_size or ndx == last_ndx:
                run_last_ndx = ndx + start_ndx
                break
        except Exception as error:
            print("An exception occurred:", type(error).__name__, "–", error)
            print(f"The exception occurred at index {ndx + start_ndx} with document: {doc}")
            skipped_doc_ids.append(doc.metadata["id"])
            print(f"skipped {len(skipped_doc_ids)} doc ids: {skipped_doc_ids}")
            continue 

    return {
        "graph_documents": graph_documents, 
        "skipped_doc_ids": skipped_doc_ids, 
        "run_last_ndx": run_last_ndx, 
        "is_done": ndx == last_ndx,
    }

## Convert Documents to Graph Documents

In [ ]:
all_graph_documents = []
skipped_doc_ids = []
start_ndx = 0
is_done = False
ids = [] 
num_docs = pd.read_parquet(
                hf_hub_download(repo_id=data_repo_id, filename=corpus_file, repo_type="dataset")
           ).shape[0]

if use_test_set_only:
    df_test = pd.read_parquet(
        hf_hub_download(repo_id=data_repo_id, filename=test_file, repo_type="dataset")
    )

    ids = df_test["relevant_passage_ids"].explode().apply(int).unique().tolist()
    num_docs = len(ids)


print(len(num_docs))

In [ ]:
# converting by batch because modal labs' free tier apps timeout after 1 hour at most
while not is_done:
    with app.run():
        results = convert_to_graph_documents.remote(
            model_name=MODEL, 
            repo_id=data_repo_id, 
            filename=corpus_file, 
            device="cuda",
            ids=ids,
            start_ndx=start_ndx, 
            batch_size=200,
        )
    
    all_graph_documents.extend(results["graph_documents"])
    skipped_doc_ids.extend(results["skipped_doc_ids"])

    run_last_ndx = results["run_last_ndx"] 
    start_ndx = run_last_ndx  + 1
    is_done = results["is_done"]


    print(f"{run_last_ndx + 1} documents processed overall",
          f"Skipped: {len(skipped_doc_ids)}",
          f"Pending: {num_docs - (run_last_ndx + 1)}", 
          sep="\n")
 

## Ingest to Neo4j

In [ ]:
graph = Neo4jGraph()

In [ ]:
# might be better to improve prompt with structured output when extracting graph elements 
# to lessen/remove need to clean/rebuild nodes and relationships
cleaned_graph_documents = utils.rebuild_graph_documents(all_graph_documents)

In [ ]:
graph.add_graph_documents(
    graph_documents=cleaned_graph_documents,
    baseEntityLabel=True,
    include_source=True
)